In [4]:
import gdspy

gdspy.current_library = gdspy.GdsLibrary()
gds_lib = gdspy.GdsLibrary(unit=1e-6, precision=1e-9)


def make_mark_cell(
    lib: gdspy.GdsLibrary,
    arm_len: float = 20.0,
    arm_w: float = 2.0,
    layer: int = 10,
):
    """
    Create an L-shaped mark centered at the origin corner (0,0) with arms extending +x and +y.
    """
    c = lib.new_cell('mark')

    # Horizontal arm: from (0,0) to (+arm_len, +arm_w)
    c.add(gdspy.Rectangle((0.0, 0.0), (arm_len, arm_w), layer=layer))
    c.add(gdspy.Rectangle((0.0, 0.0), (-arm_len, -arm_w), layer=layer))
    c.add(gdspy.Rectangle((0.0, 0.0), (arm_w, arm_len), layer=layer))
    c.add(gdspy.Rectangle((0.0, 0.0), (-arm_w, -arm_len), layer=layer))
    return c


cell = gds_lib.new_cell('markers')

align = make_mark_cell(gds_lib)

ref = gdspy.CellReference(align, (100, 100), rotation=0)
cell.add(ref)
ref = gdspy.CellReference(align, (-100, -100), rotation=0)
cell.add(ref)
ref = gdspy.CellReference(align, (100, -100), rotation=0)
cell.add(ref)
ref = gdspy.CellReference(align, (-100, 100), rotation=0)
cell.add(ref)

gds_lib.write_gds('GDS/markers.gds')


In [ ]:
import gdspy

gdspy.current_library = gdspy.GdsLibrary()
gds_lib = gdspy.GdsLibrary(unit=1e-6, precision=1e-9)

end_beam = 2000
beam_cell = gds_lib.new_cell('beam_rect')
beam_cell.add(gdspy.Rectangle((0, 0), (end_beam, 10), layer=0))

cell = gds_lib.new_cell('beams')

N = 10
for i in range(N):
    y0 = i * 20
    ref = gdspy.CellReference(beam_cell, (0, y0))  # also fix: offset per copy
    cell.add(ref)

align = make_mark_cell(gds_lib)

ref = gdspy.CellReference(align, (-100, -100), rotation=0)
cell.add(ref)
ref = gdspy.CellReference(align, (-100, y0+100), rotation=90)
cell.add(ref)
ref = gdspy.CellReference(align, (end_beam+100, -100), rotation=90)
cell.add(ref)
ref = gdspy.CellReference(align, (end_beam+100, y0+100), rotation=0)
cell.add(ref)

gds_lib.write_gds('GDS/beams_with_markers.gds')

# TEST BEAM EDGE COUPLING

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from main_code.cavity import Cavity
from main_code.simulation import Cavity_simulation
from main_code.bandstructure_class import BandStructureSim

import tidy3d as td
C0 = td.constants.C_0

n_cells = {
    "N_left_taper":    5,
    "N_left_mirror":  10,
    "N_defect":       30,   # odd → includes a central cell
    "N_right_mirror": 20,
    "N_right_taper":   1,
}

parameters = {
    "parameters_taper_left":    {"lattice": 0.48, "hole_params": np.array([0.01, 0.01])},
    "parameters_mirrors_left":  {"lattice": 0.5, "hole_params": np.array([0.25, 0.25])},
    "parameters_defect":        {"lattice": 0.47,  "hole_params": np.array([0.25, 0.25])},
    "parameters_mirrors_right": {"lattice": 0.5, "hole_params": np.array([0.25, 0.25])},
    "parameters_taper_right":   {"lattice": 0.48, "hole_params": np.array([0.01, 0.01])},
}

wavelength = 1.324
freq0 = C0 / wavelength
fwidth = freq0 * 0.01
width = 1
thickness = 0.25

context = {
    "freq0":          freq0,       # centre frequency (1/µm)
    "fwidth":         fwidth,        # bandwidth
    "thickness":      thickness,       # slab thickness (µm)
    "width":          width,       # waveguide width (µm)
    "polarization":   "Ey",       # dipole polarization (TE-like)
    "medium":         2.0,          # material name (file-based dispersion) or refractive index
    "mode":           "dielectric",
    "sidewall_angle":  0,
    "geometry":       "ellipse",  # hole shape: 'ellipse' or 'square'
}

cavity1 = Cavity(n_cells=n_cells, parameters=parameters, context=context)

In [9]:
import gdspy

gdspy.current_library = gdspy.GdsLibrary()
gds_lib = gdspy.GdsLibrary(unit=1e-6, precision=1e-9)

cell = gds_lib.new_cell('beam_with_holes')

start_beam = cavity1.beam_layout['positions'][0] - 1
end_beam = cavity1.beam_layout['positions'][-1] + 1  
length_x = end_beam - start_beam
length_y = width

beam = gdspy.Rectangle((start_beam, -length_y/2), (end_beam, length_y/2), layer=1)
triangle = gdspy.Polygon([(end_beam, length_y/2),(end_beam + 10, 0),(end_beam, -length_y/2)], layer=1)


holes = []

for i, x in enumerate(cavity1.beam_layout['positions']):
    rx = float(cavity1.beam_layout['hole_params'][i, 0])/2
    ry = float(cavity1.beam_layout['hole_params'][i, 1])/2

    hole = gdspy.Round((x, 0.0), radius = (rx, ry), number_of_points=32, layer=0)    
    holes.append(hole)
    #cell.add(hole)
    
cell.add(beam)
cell.add(triangle)
cell.add(holes)

#inverted_geometry = gdspy.boolean(beam, holes, 'not', layer=1, datatype=0)
#cell.add(inverted_geometry)

length_waveguide = 100
length_taper = 4000

wgd = gdspy.Rectangle((-length_waveguide, -length_y/2), (start_beam, length_y/2), layer=2)
cell.add(wgd)

poli = gdspy.Polygon([(-length_taper, -thickness/2),(-length_taper, thickness/2),(-length_waveguide,length_y/2),(-length_waveguide,-length_y/2)], layer=2)
cell.add(poli)


array_cell = gds_lib.new_cell('beam_with_holes_array')

N = 10           # number of repeats
dy = 12.7         # spacing in y (in your library units; here it's microns if unit=1e-6)

'''text_layer = 10
text_dtype = 0
text_height = 3.0   # text size in your units (microns if unit=1e-6)

# Where to place the number relative to each repeated structure:
x_label = -5      # pick something in your layout
y_label_offset = 2  # relative to each copy's origin'''

for k in range(N):
    y0 = k * dy
    # Place the structure
    array_cell.add(gdspy.CellReference(cell, origin=(0, y0)))

'''
    # Add the number label
    label = gdspy.Text(
        str(k),                      # or str(k+1) if you want 1..N
        text_height,
        position=(x_label, y0 + y_label_offset),
        layer=text_layer,
        datatype=text_dtype
    )
    array_cell.add(label)
'''


opening_end = 200
opening = gdspy.Rectangle((start_beam, -length_y/2 - 100), (opening_end, y0+100), layer=3)
array_cell.add(opening)

align = make_mark_cell(gds_lib)

shift = 50

ref = gdspy.CellReference(align, (-length_taper-shift, -shift), rotation=0)
array_cell.add(ref)
ref = gdspy.CellReference(align, (-length_taper-shift, y0+shift), rotation=90)
array_cell.add(ref)
ref = gdspy.CellReference(align, (opening_end + shift, -shift), rotation=90)
array_cell.add(ref)
ref = gdspy.CellReference(align, (opening_end + shift, y0+shift), rotation=0)
array_cell.add(ref)

gds_lib.write_gds('GDS/test_allison_flow_not_punched_gdspy.gds')


In [6]:
import gdsfactory as gf
import numpy as np
from functools import partial
import sys
sys.path.append('/Users/luchito/Documents/github/layout')
from layout.core.routing import route0
from layout.components.gratings import grating

gf.gpdk.PDK.activate()
gf.clear_cache()

# --- Shared mark cell (created once) ---
_mark_cell = None

def get_mark_cell(arm_len=20.0, arm_w=2.0, layer=(10, 0)):
    global _mark_cell
    if _mark_cell is None:
        _mark_cell = gf.Component("mark")
        _mark_cell.add_polygon([(0, 0), (arm_len, 0), (arm_len, arm_w), (0, arm_w)], layer=layer)
        _mark_cell.add_polygon([(0, 0), (-arm_len, 0), (-arm_len, -arm_w), (0, -arm_w)], layer=layer)
        _mark_cell.add_polygon([(0, 0), (arm_w, 0), (arm_w, arm_len), (0, arm_len)], layer=layer)
        _mark_cell.add_polygon([(0, 0), (-arm_w, 0), (-arm_w, -arm_len), (0, -arm_len)], layer=layer)
    return _mark_cell

# --- Edge-coupled beam ---
def beam_with_holes_cell(positions, hole_params, width=1.0, thickness=0.5,
                         length_waveguide=100, length_taper=1000):
    c = gf.Component("beam_edge")
    positions_arr = np.array(positions)
    hole_params_arr = np.array(hole_params)
    start_beam = positions_arr[0] - 1
    end_beam = positions_arr[-1] + 1
    length_y = width

    c.add_polygon(
        [(start_beam, -length_y / 2), (end_beam, -length_y / 2),
         (end_beam, length_y / 2), (start_beam, length_y / 2)],
        layer=(1, 0),
    )
    c.add_polygon(
        [(end_beam, length_y / 2), (end_beam + 10, 0), (end_beam, -length_y / 2)],
        layer=(1, 0),
    )
    for i, x in enumerate(positions_arr):
        rx = float(hole_params_arr[i, 0]) / 2
        ry = float(hole_params_arr[i, 1]) / 2
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        pts = [(x + rx * np.cos(a), ry * np.sin(a)) for a in angles]
        c.add_polygon(pts, layer=(0, 0))

    c.add_polygon(
        [(-length_waveguide, -length_y / 2), (start_beam, -length_y / 2),
         (start_beam, length_y / 2), (-length_waveguide, length_y / 2)],
        layer=(2, 0),
    )
    c.add_polygon(
        [(-length_taper, -thickness / 2), (-length_taper, thickness / 2),
         (-length_waveguide, length_y / 2), (-length_waveguide, -length_y / 2)],
        layer=(2, 0),
    )
    return c

def beam_array_edge(positions, hole_params, width=1.0, thickness=0.5,
                    length_waveguide=100, length_taper=1000, N=10, dy=12.7):
    c = gf.Component("edge_array")
    beam_cell = beam_with_holes_cell(
        positions=positions, hole_params=hole_params,
        width=width, thickness=thickness,
        length_waveguide=length_waveguide, length_taper=length_taper,
    )
    pos_arr = np.array(positions)
    start_beam = pos_arr[0] - 1
    opening_end = 200

    for k in range(N):
        c.add_ref(beam_cell).move((0, k * dy))

    y_last = (N - 1) * dy
    c.add_polygon(
        [(start_beam, -width / 2 - 100), (opening_end, -width / 2 - 100),
         (opening_end, y_last + 100), (start_beam, y_last + 100)],
        layer=(3, 0),
    )

    align = get_mark_cell()
    shift = 50
    for (tx, ty), rot in [
        ((-length_taper - shift, -shift), 0),
        ((-length_taper - shift, y_last + shift), 90),
        ((opening_end + shift, -shift), 90),
        ((opening_end + shift, y_last + shift), 0),
    ]:
        ref = c.add_ref(align)
        ref.rotate(rot)
        ref.move((tx, ty))

    return c

# --- Generate edge coupling GDS ---
top_edge = beam_array_edge(
    positions=tuple(cavity1.beam_layout['positions'].tolist()),
    hole_params=tuple(map(tuple, cavity1.beam_layout['hole_params'].tolist())),
    width=width,
    thickness=thickness,
    N=5,
    dy=12.7,
)
top_edge.write_gds('GDS/test_edge_coupling_1000.gds')

PosixPath('GDS/test_edge_coupling_1000.gds')

# TEST BEAM GRATING COUPLING

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from main_code.cavity import Cavity
from main_code.simulation import Cavity_simulation
from main_code.bandstructure_class import BandStructureSim

import tidy3d as td
C0 = td.constants.C_0

n_cells = {
    "N_left_taper":    5,
    "N_left_mirror":  10,
    "N_defect":       30,   # odd → includes a central cell
    "N_right_mirror": 10,
    "N_right_taper":   5,
}

parameters = {
    "parameters_taper_left":    {"lattice": 0.48, "hole_params": np.array([0.01, 0.01])},
    "parameters_mirrors_left":  {"lattice": 0.5, "hole_params": np.array([0.25, 0.25])},
    "parameters_defect":        {"lattice": 0.47,  "hole_params": np.array([0.25, 0.25])},
    "parameters_mirrors_right": {"lattice": 0.5, "hole_params": np.array([0.25, 0.25])},
    "parameters_taper_right":   {"lattice": 0.48, "hole_params": np.array([0.01, 0.01])},
}

wavelength = 1.324
freq0 = C0 / wavelength
fwidth = freq0 * 0.01
width = 1
thickness = 0.25

context = {
    "freq0":          freq0,       # centre frequency (1/µm)
    "fwidth":         fwidth,        # bandwidth
    "thickness":      thickness,       # slab thickness (µm)
    "width":          width,       # waveguide width (µm)
    "polarization":   "Ey",       # dipole polarization (TE-like)
    "medium":         2.0,          # material name (file-based dispersion) or refractive index
    "mode":           "dielectric",
    "sidewall_angle":  0,
    "geometry":       "ellipse",  # hole shape: 'ellipse' or 'square'
}

cavity2 = Cavity(n_cells=n_cells, parameters=parameters, context=context)

In [10]:
import gdsfactory as gf
import numpy as np
from functools import partial
import sys
sys.path.append('/Users/luchito/Documents/github/layout')
from layout.core.routing import route0
from layout.components.gratings import grating

gf.gpdk.PDK.activate()
gf.clear_cache()

# --- Shared mark cell (created once) ---
_mark_cell = None

def get_mark_cell(arm_len=20.0, arm_w=2.0, layer=(10, 0)):
    global _mark_cell
    if _mark_cell is None:
        _mark_cell = gf.Component("mark")
        _mark_cell.add_polygon([(0, 0), (arm_len, 0), (arm_len, arm_w), (0, arm_w)], layer=layer)
        _mark_cell.add_polygon([(0, 0), (-arm_len, 0), (-arm_len, -arm_w), (0, -arm_w)], layer=layer)
        _mark_cell.add_polygon([(0, 0), (arm_w, 0), (arm_w, arm_len), (0, arm_len)], layer=layer)
        _mark_cell.add_polygon([(0, 0), (-arm_w, 0), (-arm_w, -arm_len), (0, -arm_len)], layer=layer)
    return _mark_cell

# --- Grating-coupled beam ---
def beam_with_holes_and_gratings(positions, hole_params, width=1.0, thickness=0.5,
                                  dx=180, dy_grating=30, bend_radius=40):
    c = gf.Component("beam_grating")
    positions_arr = np.array(positions)
    hole_params_arr = np.array(hole_params)
    start_beam = positions_arr[0] - 1
    end_beam = positions_arr[-1] + 1
    length_y = width

    c.add_polygon(
        [(start_beam, -length_y / 2), (end_beam, -length_y / 2),
         (end_beam, length_y / 2), (start_beam, length_y / 2)],
        layer=(1, 0),
    )
    for i, x in enumerate(positions_arr):
        rx = float(hole_params_arr[i, 0]) / 2
        ry = float(hole_params_arr[i, 1]) / 2
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        pts = [(x + rx * np.cos(a), ry * np.sin(a)) for a in angles]
        c.add_polygon(pts, layer=(0, 0))

    c.add_port(name="left", center=(start_beam, 0), width=width, orientation=180, layer=(1, 0))
    c.add_port(name="right", center=(end_beam, 0), width=width, orientation=0, layer=(1, 0))

    center_x = (start_beam + end_beam) / 2
    g1 = c.add_ref_off_grid(grating(layer=1, width=width)).rotate(135).move((center_x - dx / 2, dy_grating))
    g2 = c.add_ref_off_grid(grating(layer=1, width=width)).rotate(45).move((center_x + dx / 2, dy_grating))

    route0(c, c.ports["left"], g1.ports["o1"], taper_length=0,
           bend=partial(gf.components.bend_euler_all_angle, radius=bend_radius, p=0.1))
    route0(c, c.ports["right"], g2.ports["o1"], taper_length=0,
           bend=partial(gf.components.bend_euler_all_angle, radius=bend_radius, p=0.1))
    return c

def beam_array_grating(positions, hole_params, width=1.0, thickness=0.5,
                       N=10, dy=50.0):
    c = gf.Component("grating_array")
    beam_cell = beam_with_holes_and_gratings(
        positions=positions, hole_params=hole_params,
        width=width, thickness=thickness,
    )
    pos_arr = np.array(positions)
    start_beam = pos_arr[0] - 1
    end_beam = pos_arr[-1] + 1

    for k in range(N):
        c.add_ref(beam_cell).move((0, k * dy))

    y_bottom = -width / 2 - 100
    y_top = (N - 1) * dy + width / 2 + 100
    shift = 10
    c.add_polygon(
        [(start_beam - shift, y_bottom), (end_beam + shift, y_bottom),
         (end_beam + shift, y_top), (start_beam - shift, y_top)],
        layer=(3, 0),
    )

    align = get_mark_cell()
    margin = 50
    for cx, cy, rot in [
        (start_beam - shift - margin, y_bottom - margin, 0),
        (end_beam + shift + margin, y_bottom - margin, 90),
        (end_beam + shift + margin, y_top + margin, 0),
        (start_beam - shift - margin, y_top + margin, 90),
    ]:
        ref = c.add_ref(align)
        ref.rotate(rot)
        ref.move((cx, cy))

    return c

# --- Generate grating coupling GDS ---
top_grating = beam_array_grating(
    positions=tuple(cavity2.beam_layout['positions'].tolist()),
    hole_params=tuple(map(tuple, cavity2.beam_layout['hole_params'].tolist())),
    width=width,
    thickness=thickness,
    N=5,
    dy=50.0,
)
top_grating.write_gds('GDS/test_grating_coupling_5_symmetric.gds')

DInstancePorts(ports=[DPort(self.name='o1', self.width=1.0, trans=r180 *1 -23.8,0, layer=WG (1/0), port_type=optical), DPort(self.name='o2', self.width=10.0, trans=r0 *1 0.7,0, layer=WG (1/0), port_type=vertical_te)])


PosixPath('GDS/test_grating_coupling_5_symmetric.gds')

# EDGE AND GRATING COMBINED

In [12]:
import gdsfactory as gf
import numpy as np
from functools import partial
import sys
sys.path.append('/Users/luchito/Documents/github/layout')
from layout.core.routing import route0
from layout.components.gratings import grating

gf.gpdk.PDK.activate()
gf.clear_cache()

# --- Shared mark cell (created once) ---
_mark_cell = None

def get_mark_cell(arm_len=20.0, arm_w=2.0, layer=(10, 0)):
    global _mark_cell
    if _mark_cell is None:
        _mark_cell = gf.Component("mark")
        _mark_cell.add_polygon([(0, 0), (arm_len, 0), (arm_len, arm_w), (0, arm_w)], layer=layer)
        _mark_cell.add_polygon([(0, 0), (-arm_len, 0), (-arm_len, -arm_w), (0, -arm_w)], layer=layer)
        _mark_cell.add_polygon([(0, 0), (arm_w, 0), (arm_w, arm_len), (0, arm_len)], layer=layer)
        _mark_cell.add_polygon([(0, 0), (-arm_w, 0), (-arm_w, -arm_len), (0, -arm_len)], layer=layer)
    return _mark_cell

# --- Edge-coupled beam (created once, referenced many times) ---
def beam_with_holes_cell(positions, hole_params, width=1.0, thickness=0.5,
                         length_waveguide=100, length_taper=1000):
    c = gf.Component("beam_edge")
    positions_arr = np.array(positions)
    hole_params_arr = np.array(hole_params)
    start_beam = positions_arr[0] - 1
    end_beam = positions_arr[-1] + 1
    length_y = width

    c.add_polygon(
        [(start_beam, -length_y / 2), (end_beam, -length_y / 2),
         (end_beam, length_y / 2), (start_beam, length_y / 2)],
        layer=(1, 0),
    )
    c.add_polygon(
        [(end_beam, length_y / 2), (end_beam + 10, 0), (end_beam, -length_y / 2)],
        layer=(1, 0),
    )
    for i, x in enumerate(positions_arr):
        rx = float(hole_params_arr[i, 0]) / 2
        ry = float(hole_params_arr[i, 1]) / 2
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        pts = [(x + rx * np.cos(a), ry * np.sin(a)) for a in angles]
        c.add_polygon(pts, layer=(0, 0))

    c.add_polygon(
        [(-length_waveguide, -length_y / 2), (start_beam, -length_y / 2),
         (start_beam, length_y / 2), (-length_waveguide, length_y / 2)],
        layer=(2, 0),
    )
    c.add_polygon(
        [(-length_taper, -thickness / 2), (-length_taper, thickness / 2),
         (-length_waveguide, length_y / 2), (-length_waveguide, -length_y / 2)],
        layer=(2, 0),
    )
    return c

# --- Grating-coupled beam (created once, referenced many times) ---
def beam_with_holes_and_gratings(positions, hole_params, width=1.0, thickness=0.5,
                                  dx=180, dy_grating=30, bend_radius=40):
    c = gf.Component("beam_grating")
    positions_arr = np.array(positions)
    hole_params_arr = np.array(hole_params)
    start_beam = positions_arr[0] - 1
    end_beam = positions_arr[-1] + 1
    length_y = width

    c.add_polygon(
        [(start_beam, -length_y / 2), (end_beam, -length_y / 2),
         (end_beam, length_y / 2), (start_beam, length_y / 2)],
        layer=(1, 0),
    )
    for i, x in enumerate(positions_arr):
        rx = float(hole_params_arr[i, 0]) / 2
        ry = float(hole_params_arr[i, 1]) / 2
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        pts = [(x + rx * np.cos(a), ry * np.sin(a)) for a in angles]
        c.add_polygon(pts, layer=(0, 0))

    c.add_port(name="left", center=(start_beam, 0), width=width, orientation=180, layer=(1, 0))
    c.add_port(name="right", center=(end_beam, 0), width=width, orientation=0, layer=(1, 0))

    center_x = (start_beam + end_beam) / 2
    g1 = c.add_ref_off_grid(grating(layer=1, width=width)).rotate(135).move((center_x - dx / 2, dy_grating))
    g2 = c.add_ref_off_grid(grating(layer=1, width=width)).rotate(45).move((center_x + dx / 2, dy_grating))

    route0(c, c.ports["left"], g1.ports["o1"], taper_length=0,
           bend=partial(gf.components.bend_euler_all_angle, radius=bend_radius, p=0.1))
    route0(c, c.ports["right"], g2.ports["o1"], taper_length=0,
           bend=partial(gf.components.bend_euler_all_angle, radius=bend_radius, p=0.1))
    return c

# --- Edge array ---
def beam_array_edge(positions, hole_params, width=1.0, thickness=0.5,
                    length_waveguide=100, length_taper=1000, N=10, dy=12.7):
    c = gf.Component("edge_array")

    # Build the beam cell once, reference it N times
    beam_cell = beam_with_holes_cell(
        positions=positions, hole_params=hole_params,
        width=width, thickness=thickness,
        length_waveguide=length_waveguide, length_taper=length_taper,
    )

    pos_arr = np.array(positions)
    start_beam = pos_arr[0] - 1
    opening_end = 200

    for k in range(N):
        c.add_ref(beam_cell).move((0, k * dy))

    y_last = (N - 1) * dy
    c.add_polygon(
        [(start_beam, -width / 2 - 100), (opening_end, -width / 2 - 100),
         (opening_end, y_last + 100), (start_beam, y_last + 100)],
        layer=(3, 0),
    )

    align = get_mark_cell()
    shift = 50
    for (tx, ty), rot in [
        ((-length_taper - shift, -shift), 0),
        ((-length_taper - shift, y_last + shift), 90),
        ((opening_end + shift, -shift), 90),
        ((opening_end + shift, y_last + shift), 0),
    ]:
        ref = c.add_ref(align)
        ref.rotate(rot)
        ref.move((tx, ty))

    return c

# --- Grating array ---
def beam_array_grating(positions, hole_params, width=1.0, thickness=0.5,
                       N=10, dy=50.0):
    c = gf.Component("grating_array")

    # Build the grating beam cell once, reference it N times
    beam_cell = beam_with_holes_and_gratings(
        positions=positions, hole_params=hole_params,
        width=width, thickness=thickness,
    )

    pos_arr = np.array(positions)
    start_beam = pos_arr[0] - 1
    end_beam = pos_arr[-1] + 1

    for k in range(N):
        c.add_ref(beam_cell).move((0, k * dy))

    y_bottom = -width / 2 - 100
    y_top = (N - 1) * dy + width / 2 + 100
    shift = 10
    c.add_polygon(
        [(start_beam - shift, y_bottom), (end_beam + shift, y_bottom),
         (end_beam + shift, y_top), (start_beam - shift, y_top)],
        layer=(3, 0),
    )

    align = get_mark_cell()
    margin = 50
    for cx, cy, rot in [
        (start_beam - shift - margin, y_bottom - margin, 0),
        (end_beam + shift + margin, y_bottom - margin, 90),
        (end_beam + shift + margin, y_top + margin, 0),
        (start_beam - shift - margin, y_top + margin, 90),
    ]:
        ref = c.add_ref(align)
        ref.rotate(rot)
        ref.move((cx, cy))

    return c

# --- Combined top cell ---
def combined_layout(positions1, hole_params1, positions2, hole_params2,
                    width=1.0, thickness=0.5, N_edge=10, dy_edge=12.7,
                    N_grating=10, dy_grating=50.0, gap=200.0):
    c = gf.Component("TOP")

    # Edge-coupled array at the bottom
    edge = beam_array_edge(
        positions=positions1, hole_params=hole_params1,
        width=width, thickness=thickness, N=N_edge, dy=dy_edge,
    )
    c.add_ref(edge).move((0, 0))

    # Grating-coupled array above it
    edge_top = (N_edge - 1) * dy_edge + 100
    grating_arr = beam_array_grating(
        positions=positions2, hole_params=hole_params2,
        width=width, thickness=thickness, N=N_grating, dy=dy_grating,
    )
    c.add_ref(grating_arr).move((0, edge_top + gap))

    return c

# --- Usage ---
top = combined_layout(
    positions1=tuple(cavity1.beam_layout['positions'].tolist()),
    hole_params1=tuple(map(tuple, cavity1.beam_layout['hole_params'].tolist())),
    positions2=tuple(cavity2.beam_layout['positions'].tolist()),
    hole_params2=tuple(map(tuple, cavity2.beam_layout['hole_params'].tolist())),
    width=width,
    thickness=thickness,
    N_edge=5,
    dy_edge=12.7,
    N_grating=2,
    dy_grating=50.0,
    gap=200.0,
)
top.write_gds('GDS/test_combined_edge_and_grating_2_1000.gds')

DInstancePorts(ports=[DPort(self.name='o1', self.width=1.0, trans=r180 *1 -23.8,0, layer=WG (1/0), port_type=optical), DPort(self.name='o2', self.width=10.0, trans=r0 *1 0.7,0, layer=WG (1/0), port_type=vertical_te)])


PosixPath('GDS/test_combined_edge_and_grating_2_1000.gds')